# India LGD district codes (current vintage)
Uses only the Python standard library.

1. Map `state` + `district` through `data/india-lgd-district.json` (Ahmednagar and Ahilyanagar share LGD 466).
2. Inner-join to an LGD-coded population file.
3. Unmatched names are **dropped**. Census 2011 codes are not LGD codes.

Result grain is **current LGD district**, vintage 2026-07.


In [ ]:
import csv, json, re
from collections import defaultdict
from pathlib import Path

root = Path(".")
cross = json.loads((root / "data/india-lgd-district.json").read_text())
norm = lambda s: re.sub(r"[^A-Z0-9]+", " ", str(s).upper()).strip()

def lgd(state, district):
    return cross["byDistrict"].get(f"{norm(state)}|{norm(district)}")

named = list(csv.DictReader((root / "data/samples/india-lgd-name-sample.csv").open()))
by_lgd = defaultdict(float)
dropped = 0
for row in named:
    hit = lgd(row["state"], row["district"])
    if not hit:
        dropped += 1
        continue
    by_lgd[(hit["lgd"], int(row["year"]))] += float(row["scheme_value"])
assert dropped == 1, dropped
assert ("466", 2022) in by_lgd

pop = {}
with (root / "data/samples/india-lgd-code-sample.csv").open() as fh:
    for row in csv.DictReader(fh):
        pop[(row["lgd_code"], int(row["year"]))] = int(row["population"])

joined = []
for key, value in sorted(by_lgd.items()):
    if key not in pop:
        continue
    joined.append((key[0], key[1], value, pop[key]))
assert joined
census_as_lgd = [k for k in pop if k[0] == "522"]
assert not census_as_lgd, "census 2011 code 522 must not appear as an LGD key"
print("lgd year scheme_value population")
for row in joined:
    print(*row)
print(f"dropped unmatched names: {dropped}; {len(joined)} LGD-year rows")
